In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    EmpfaengerID,
    drop_duplicate_columns,
    common_translate,
)

### Target Population Filtering

The patients in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
rec = data["recipient_et_id_et"].copy()
targetpop = pd.read_parquet(targetpop_data)
data = data[rec.isin(targetpop["recipient_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of recipients in the data ({rec.nunique()}) and target population ({targetpop["recipient_et_id_et"].nunique()})
            to {rec[rec.isin(targetpop["recipient_et_id_et"])].nunique()} in the processed data.
        """
    )
)
del targetpop, rec

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

Only {term}`ET` data is in this file, so no data processing was necessary at this step (see [](general:ic)).

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column which differentiates between different tests (see [](general:rf)). We kept all test results.

In [ ]:
display_long_data_doc(
    data,
    [
        "recipient_et_id_et",
    ],
    "sampling_date",
    None,
)

### Unit Conversions


We applied common translations. No furhter steps were necessary. (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

### Consolidating Columns

No consolidation was necessary. (see [](general:crc))

## Intermediate Dataset

For this longitudinal dataset we recommend the `sampling_date` column as the time axis.

In [ ]:
indcols = ["recipient_et_id_et"]
data = data.sort_index(axis=1).sort_values(["sampling_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
labtestvals = ["not tested", "positive", "negative"]


class EmpfaengerVirologie(EmpfaengerID):
    cytomegalovirus_igg: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="CMV IgG Antibodies",
        description="Were cytomegalovirus IgG antibodies present?",
        isin=labtestvals,
    )
    cytomegalovirus_igm: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="CMV IgM Antibodies",
        description="Were cytomegalovirus IgG antibodies present?",
        isin=labtestvals,
    )
    enter_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Data Entry Date",
        description="When was the data entered?",
    )
    epstein_barr_virus_igg: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="EBV IgG Antibodies",
        description="Were Epstein Barr virus IgG antibodies present?",
        isin=labtestvals,
    )
    hepatitis_b_core_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B Core Antibodies",
        description="Were Hepatitis B core antibodies present?",
        isin=labtestvals,
    )
    hepatitis_b_immunized: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B Immunization",
        description="Was the patient either vaccinated or became immunized against hepatits B?",
        isin=["yes", "no", "unknown"],
    )
    hepatitis_b_surface_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B Surface Antibodies",
        description="Were Hepatitis B surface antibodies present?",
        isin=labtestvals,
    )
    hepatitis_c_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis C Antibodies Detected",
        description="Were Hepatitis C surface antibodies present?",
        isin=labtestvals,
    )
    hiv_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HIV Antibodies",
        description="Were surface antibodies present?",
        isin=labtestvals,
    )
    hiv_antigens: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HIV Antigens",
        description="Were HIV antigens present?",
        isin=labtestvals,
    )
    sampling_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Sampling date",
        description="When was the sample taken?",
    )
    syphilis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Syphilis",
        description="Was syphilis diagnosed?",
        isin=labtestvals,
    )
    toxoplasmosis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Toxoplasmosis",
        description="Was toxoplasmosis diagnosed?",
        isin=labtestvals,
    )

    class Config:
        title = "Recipient Virology Dataset"
        description = "Each row represents a virologic test performed for a (potential) recipient. The data is based on the 'empfaenger_virology.csv' file. It contains data from the ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(EmpfaengerVirologie, data)

In [ ]:
EmpfaengerVirologie.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    EmpfaengerVirologie.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)